In [ ]:
!pip install ndvi2gif

In [ ]:
import geemap
import ee
from ndvi2gif import NdviSeasonality

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-digdgeografo')

In [ ]:
# Test rápido
from ndvi2gif import NdviSeasonality

# Test S3
s2_test = NdviSeasonality(sat='S2', index='ndvi')
print(s2_test.get_available_indices('S2'))

In [ ]:
Map = geemap.Map()
Map

In [ ]:
roi = Map.draw_last_feature

In [ ]:
# Doñana NDVI Stats
myclass = NdviSeasonality(roi=roi, sat='S2',
                          key='median',
                          #percentile=75,
                          periods=12,
                          start_year=2020,
                          end_year=2023,
                          index='ndvi',
                          cloud_filter=False)

Landsat = myclass.get_year_composite()
vizParams = {'bands': ['february', 'april', 'july'], 'min': 0.1, 'max':0.8}

Map.addLayer(Landsat, vizParams, 'Median')

In [ ]:
myclass.get_export(
    scale=30,
    crs="EPSG:32629"
)

In [ ]:
myclass.get_export_single(
    image=Landsat.max(),
    name='max_power.tif',
    scale=10,
    crs="EPSG:32629"
)

In [ ]:
roi_shapefile = 'path/to/your/shapefile.shp'
roi_geojson = 'path/to/your/file.geojson'

roi_coords = ee.Geometry.Rectangle([-6.766, 36.776, -5.867, 37.202])  # [xMin, yMin, xMax, yMax]

roi_landsat = 'wrs:202,034'  # Format: 'wrs:path,row'
# Example: Path 202, Row 034 covers parts of Spain

roi_s2_tile = 's2:T30TYN'  # Format: 's2:TILE_ID'
# Example: T30TYN covers parts of Central Spain

# Use European Long-Term Ecosystem Research sites
roi_elter = 'deimsid:ab8278e6-0b71-4b36-a6d2-e8f34aa3df30'  # Format: 'deimsid:SITE_ID'
# Note: it could required 'deims' package: pip install deims in case conda couldn't make the installation

In [ ]:
Map2 = geemap.Map()
Map2

In [ ]:
deims = 'deimsid:https://deims.org/bcbc866c-3f4f-47a8-bbbc-0a93df6de7b2'

# Doñana NDVI Stats
myclass = NdviSeasonality(roi=deims, sat='S2',
                          key='percentile',
                          percentile=75,
                          periods=12,
                          start_year=2020,
                          end_year=2023,
                          index='ndvi',
                          cloud_filter=False)

Landsat = myclass.get_year_composite()
vizParams = {'bands': ['february', 'may', 'september'], 'min': 0.1, 'max':0.8}

Map2.addLayer(Landsat.median(), vizParams, 'Median')

In [ ]:
# Mediana MNDWI de toda la serie Landsat (L4-5-7-8-9 fusionados)
roi_landsat = 'wrs:202,034'
proc = NdviSeasonality(
    roi=roi_landsat,
    sat='Landsat',
    index='mndwi',
    key='max',
    #percentile=95,
    periods=1,
    start_year=1984,
    end_year=2025,
    cloud_filter=True,
    max_cloud_cover=10
)

# 1 imagen por año (banda 'p1'); después hacemos la mediana interanual
coll = proc.get_year_composite()
mndwi_alltime_max = coll.median()

# Visualización (rango típico MNDWI ~[-1, 1]; agua suele ser positivo)
viz = {'min': 0.2, 'max': 0.6}
Map2.addLayer(mndwi_alltime_max, viz, 'Landsat MNDWI Max (1998-1990)')

In [ ]:
# Máxima MNDWI de INVIERNO (Landsat, toda la serie histórica)
roi_landsat = 'wrs:202,034'

proc = NdviSeasonality(
    roi= roi_landsat,
    sat='Landsat',
    index='mndwi',
    key='percentile',
    percentile = 95,
    periods=4,         # estaciones: winter, spring, summer, autumn
    start_year=2008,
    end_year=2011,
    cloud_filter=True,
    max_cloud_cover=10
)

# Colección: 1 imagen por año, con bandas ['winter','spring','summer','autumn']
coll = proc.get_year_composite()

# Máximo interanual SOLO de la banda 'winter'
winter_max_alltime = coll.select('summer').median()#.rename('MNDWI_winter_max_alltime')

winter_p95_alltime = coll.select('summer') \
    .reduce(ee.Reducer.percentile([95])) \
    .rename('MNDWI_winter_p95_alltime')

# Visualizar
viz = {'min': -0.2, 'max': 0.6}
Map2.addLayer(winter_p95_alltime, viz, 'Landsat MNDWI Winter MAX (All-time)')

In [ ]:
import ee, os, pathlib, shutil

# 1) Borra credenciales para forzar reauth
cred_path = pathlib.Path.home() / ".config" / "earthengine" / "credentials"
if cred_path.exists():
    shutil.rmtree(cred_path.parent)  # borra la carpeta earthengine completa

# 2) Autentica con scopes (usa la UI del notebook)
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/drive',
    ]
)

# 3) Inicializa
ee.Initialize(project='ee-digdgeografo')

In [ ]:
task = proc.export_to_drive(
    image=winter_p95_alltime,
    description="mndwi_winter_p95_alltime",
    folder="ndvi2gif_exports",
    file_format="GeoTIFF",
    crs="EPSG:32629",
    scale=30,
    max_pixels=int(1e13),
    file_dimensions=8192,    # opcional para trocear
    clip_region=True        # opcional para recortar
    #format_options={"cloudOptimized": True},  # <-- sin 'compression'
)
print("Task:", task.id)

In [ ]:
task = proc.export_to_asset(
    image=winter_p95_alltime,  # MNDWI = float
    asset_id="users/ee-digdgeografo/assets/mndwi_winter_p95_alltime",
    description="mndwi_winter_p95_alltime",
    region=proc.roi,
    crs="EPSG:32630",
    scale=30,
    pyramiding_policy={"MNDWI_winter_p95_alltime": "mean"},  # <- float
    clip_region=True,
    overwrite=True
)